# Topic 04 — Data Cleaning & Missing Data

> **Investigation milestone:** This is where the case cracks open. Aurora's
> `channel` and `status` have inconsistent casing (so totals are split),
> `customers` has duplicate rows (inflating customer counts), prices are
> missing, and dates are text. Until you clean, every number is a lie.

**Time split: 20% reading · 80% in `practice.ipynb`. Cleaning is ~80% of real analyst work.**

---

## A repeatable cleaning workflow

1. **Profile** — `info()`, `describe()`, `isna().sum()`, `nunique()`, `value_counts()`.
2. **Fix dtypes** — `astype`, `pd.to_datetime`, `pd.to_numeric(errors="coerce")`.
3. **Standardise categories** — strip whitespace, fix casing, map synonyms.
4. **Handle duplicates** — `duplicated`, `drop_duplicates(subset=, keep=)`.
5. **Handle missing** — decide per column: drop / fill / leave NaN.
6. **Validate** — re-profile; assert invariants.

## Missing data — `NaN`, `None`, `pd.NA`

- `NaN` is a float (`np.nan`); it propagates (`NaN + 1 = NaN`) and `NaN != NaN`.
- `None` becomes `NaN` in numeric columns, stays `None`/`NaN` in object columns.
- `pd.NA` is the newer, dtype-agnostic missing marker.

Detect & act:
```python
df.isna().sum()                 # missing per column
df["unit_price"].fillna(0)      # fill (decide if 0 is honest!)
df.dropna(subset=["email"])     # drop rows missing a key field
df["unit_price"].fillna(df["unit_price"].median())   # impute
```

> **Judgement, not reflex.** Filling missing *revenue* with 0 understates it;
> filling with the mean invents money. The right move depends on *why* it's
> missing. Investigate before you impute (Interview Q6, Q8).

## Strings & categories

```python
orders["channel"] = orders["channel"].str.strip().str.title()  # "online " -> "Online"
orders["status"]  = orders["status"].str.lower()
country_map = {"uk": "United Kingdom", "u.k.": "United Kingdom"}
customers["country"] = customers["country"].str.lower().replace(country_map)
```
Standardising casing **merges the split categories** — and Aurora's channel
revenue suddenly stops being double-counted.

## Duplicates

```python
customers.duplicated(subset=["customer_id"]).sum()        # how many dupes
customers = customers.drop_duplicates(subset=["customer_id"], keep="first")
```
Choose the `subset` deliberately: full-row dupes vs same-key dupes are different
problems. Dropping blindly can delete legitimate rows (Interview Q10).

## Wrong values / outliers

`pd.to_numeric(s, errors="coerce")` turns junk into NaN. Negative `list_price`
and 1,500-unit quantities are *impossible* — flag or clip them, don't silently keep.

## NumPy connection 🔢

`np.where(cond, a, b)` is the vectorized "if-else" for cleaning:
```python
items["unit_price"] = np.where(items["unit_price"] < 0, np.nan, items["unit_price"])
```
`np.nan` is literally a NumPy float. `isna()` mirrors `np.isnan` but also handles
object/`pd.NA`. Masking (Topic 3) is how you target the rows to fix.

## Visual learning 📊

Before/after `value_counts().plot(kind="bar")` on `channel` is the perfect
"look what cleaning did" chart — categories collapse from 6 messy to 4 clean.

---

## 🔎 Interview Lens (answer in writing)
- **Q6:** How do you choose between dropping, filling with 0, mean-filling, and leaving NaN for missing revenue?
- **Q9:** A numeric column loads as `object` — likely causes and how to confirm each?

### Recap
1. Three ways to handle a missing value, and when each is honest.
2. How does standardising casing change Aurora's channel totals?
3. Why prefer `to_numeric(errors="coerce")` over `astype(float)` on dirty data?

➡️ Open **`practice.ipynb`**.